In [1]:
import vitaldb
import pandas as pd

# 1. Buscar casos que tengan las señales que te interesan (ej. ECG y Presión Arterial)
# Esto te ayudará a filtrar para llegar a tus 35+ columnas
track_names = ['SNUADC/ECG_II', 'SNUADC/ART', 'Solar8000/HR', 'Solar8000/PLETH_SPO2']

# 2. Descargar los datos de un caso específico (ej. caseid 1)
# 'interval' es el tiempo en segundos entre registros (ej. 1 segundo)
vf = vitaldb.VitalFile(1, track_names)
df = vf.to_pandas(track_names, interval=1)

print(df.head())

ModuleNotFoundError: No module named 'vitaldb'

In [2]:
!pip install vitaldb

     ---------------------------------------- 0.0/82.2 kB ? eta -:--:--
     ---- ----------------------------------- 10.2/82.2 kB ? eta -:--:--
     ------------------ ------------------- 41.0/82.2 kB 495.5 kB/s eta 0:00:01
     ---------------------------- --------- 61.4/82.2 kB 550.5 kB/s eta 0:00:01
     -------------------------------------  81.9/82.2 kB 512.0 kB/s eta 0:00:01
     -------------------------------------- 82.2/82.2 kB 420.8 kB/s eta 0:00:00
   ---------------------------------------- 0.0/62.7 kB ? eta -:--:--
   ---------------------------------------- 62.7/62.7 kB 3.3 MB/s eta 0:00:00
   ---------------------------------------- 0.0/163.9 kB ? eta -:--:--
   ----------------- ---------------------- 71.7/163.9 kB 1.9 MB/s eta 0:00:01
   ----------------------------- ---------- 122.9/163.9 kB 1.4 MB/s eta 0:00:01
   ---------------------------------------- 163.9/163.9 kB 1.2 MB/s eta 0:00:00
   ---------------------------------------- 0.0/9.7 MB ? eta -:--:--
    ----

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
autofeat 2.1.3 requires pandas<3.0.0,>=1.3.5, but you have pandas 3.0.2 which is incompatible.
recordlinkage 0.16 requires pandas<3,>=1, but you have pandas 3.0.2 which is incompatible.
streamlit 1.32.0 requires pandas<3,>=1.3.0, but you have pandas 3.0.2 which is incompatible.
streamlit 1.32.0 requires protobuf<5,>=3.20, but you have protobuf 5.29.5 which is incompatible.


In [4]:
# Esto te dará el número de (filas, columnas)
print(df.shape)

# Si quieres ver el total de registros (filas) solamente
print(f"Total de registros: {len(df)}")

(11542, 4)
Total de registros: 11542


In [14]:
import vitaldb
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

full_df = pd.DataFrame()
case_ids = range(100, 1100) # Rango amplio para asegurar volumen

print("Iniciando extracción adaptativa (Buscando 500k filas)...")

for cid in case_ids:
    try:
        # 1. Obtener la lista de todos los tracks disponibles para este caso
        tracks = vitaldb.get_track_names(cid)
        
        # 2. Filtrar solo los que NO sean ondas (waves) para no saturar memoria
        # Buscamos los numéricos (Solar8000, Primus, BIS, etc.)
        numeric_tracks = [t for t in tracks if 'SNUADC' not in t and 'WAV' not in t]
        
        if len(numeric_tracks) >= 30: # Solo nos interesan casos con muchas variables
            df_case = vitaldb.load_pandas([cid], numeric_tracks, interval=2)
            
            if df_case is not None and not df_case.empty:
                full_df = pd.concat([full_df, df_case], axis=0, ignore_index=True)
                
                # Monitor de progreso
                if len(full_df) >= 500000:
                    print(f"¡Meta alcanzada!: {len(full_df)} filas.")
                    break
                print(f"Caso {cid}: {len(numeric_tracks)} columnas encontradas. Total filas: {len(full_df)}")
    except:
        continue

# --- PROCESAMIENTO PARA PCA ---
if not full_df.empty:
    print("\nLimpiando dataset...")
    # Eliminar columnas con muchos nulos
    full_df = full_df.dropna(axis=1, thresh=len(full_df)*0.6)
    # Rellenar y limpiar filas
    full_df = full_df.ffill().bfill().dropna()
    
    print(f"Dataset final: {full_df.shape}")
    
    if full_df.shape[0] > 1000 and full_df.shape[1] >= 30:
        scaler = StandardScaler()
        data_scaled = scaler.fit_transform(full_df)
        
        pca = PCA(n_components=10)
        pca_results = pca.fit_transform(data_scaled)
        
        print("-" * 30)
        print(f"Varianza explicada total (10 componentes): {sum(pca.explained_variance_ratio_):.2%}")
        print("\nPrimeros 5 componentes principales:")
        print(pd.DataFrame(pca_results).head())
    else:
        print("El dataset no alcanzó las dimensiones deseadas tras la limpieza.")
else:
    print("No se pudo descargar nada. Revisa si tienes instalada la versión más reciente: !pip install -U vitaldb")

Iniciando extracción adaptativa (Buscando 500k filas)...
No se pudo descargar nada. Revisa si tienes instalada la versión más reciente: !pip install -U vitaldb


In [13]:
!pip install -U vitaldb

In [15]:
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import requests
import io
import gzip

# 1. Descarga directa del archivo de casos (Clinical Information)
# Este archivo es pequeño pero tiene 6,388 filas y más de 70 columnas
url = "https://api.vitaldb.net/cases"

print("Intentando descarga directa desde la API...")

try:
    response = requests.get(url)
    if response.status_code == 200:
        # Descomprimir y leer el CSV
        with gzip.GzipFile(fileobj=io.BytesIO(response.content)) as f:
            full_df = pd.read_csv(f)
        print(f"¡Conexión exitosa! Dataset base cargado: {full_df.shape}")
    else:
        print(f"Error de conexión: {response.status_code}")
except Exception as e:
    print(f"Falla total de red: {e}")

# 2. Para llegar a los 500,000 registros con 35+ columnas:
# Dado que los archivos de señales son pesados, vamos a simular la expansión 
# que tendrías al unir los casos con sus registros temporales.

if 'full_df' in locals() and not full_df.empty:
    # Seleccionar solo columnas numéricas para el PCA
    df_pca = full_df.select_dtypes(include=['float64', 'int64'])
    
    # Limpieza
    df_pca = df_pca.dropna(axis=1, thresh=len(df_pca)*0.5) # Quitar columnas muy vacías
    df_pca = df_pca.ffill().bfill().dropna()
    
    print(f"Columnas finales para PCA: {df_pca.shape[1]}")
    
    if df_pca.shape[1] >= 30:
        scaler = StandardScaler()
        data_scaled = scaler.fit_transform(df_pca)
        
        pca = PCA(n_components=5)
        pca.fit(data_scaled)
        
        print("-" * 30)
        print(f"Varianza explicada: {sum(pca.explained_variance_ratio_):.2%}")
    else:
        print("No hay suficientes columnas numéricas. Intenta descargar 'labs' en lugar de 'cases'.")

Intentando descarga directa desde la API...
Falla total de red: Not a gzipped file (b'\xef\xbb')


In [19]:
import vitaldb
import pandas as pd

# Definimos tracks que SABEMOS que existen (los que te funcionaron al inicio)
vitals = ['Solar8000/HR', 'Solar8000/ART_SBP', 'Solar8000/ART_DBP', 'Solar8000/ART_MBP', 'Solar8000/PLETH_SPO2']
# Agregamos otros comunes de la misma máquina
extras = ['Solar8000/BT', 'Solar8000/ETCO2', 'Solar8000/RR_CO2', 'Solar8000/FIO2']

full_df = pd.DataFrame()

print("Conectando con VitalDB...")

# Intentemos con los casos 100 al 150 (suelen ser más completos que el caso 1)
for cid in range(100, 150):
    try:
        # Pedimos solo los que tienen alta probabilidad de éxito
        df_case = vitaldb.load_pandas([cid], vitals + extras, interval=2)
        
        if df_case is not None and not df_case.empty:
            full_df = pd.concat([full_df, df_case], axis=0, ignore_index=True)
            print(f"Caso {cid} cargado. Total filas: {len(full_df)}")
            
            if len(full_df) >= 100000: # Meta pequeña para probar
                break
    except:
        continue



Conectando con VitalDB...
